# csv merger, master table builder

builds two master tables that replace per-script csv loading and joining:

| output | granularity | replaces |
|---|---|---|
| `master_strains.csv` | one row per ssr per strain | `phasomeit_tract_data.csv` |
| `master_groups.csv` | one row per gene group | `phasomeit_full_group_summary.csv` + `phasomeit_eggnog_linked.tsv` |

filter `passes_all_filters == True` downstream to get the strict filtered set.

### filter flags added to both tables
| column | filter | applies to |
|---|---|---|
| `f_not_downstream` | ssr is not downstream of gene | all |
| `f_upstream_dist` | upstream ssr within 200 bp of start codon | upstream rows, true elsewhere |
| `f_frameshift` | repeat unit length not divisible by 3 | intragenic rows, true elsewhere |
| `f_cds_position` | ssr in first 60% of cds | intragenic with verified position, nan elsewhere |
| `passes_all_filters` | and of all applicable flags | nan where position unverifiable |


## 1. imports and paths


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parent

# ── Input paths ───────────────────────────────────────────────────────────────
TRACT_PATH      = PROJECT_ROOT / "Extraction" / "phasomeit_tract_data.csv"
GROUP_PATH      = PROJECT_ROOT / "Results" / "Tables" / "phasomeit_full_group_summary.csv"
EGGNOG_PATH     = PROJECT_ROOT / "Cleaned" / "Mapped_functions" / "phasomeit_eggnog_linked.tsv"

# ── Output paths ──────────────────────────────────────────────────────────────
OUT_DIR         = PROJECT_ROOT / "Results" / "Tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_STRAINS_PATH = OUT_DIR / "master_strains.csv"
MASTER_GROUPS_PATH  = OUT_DIR / "master_groups.csv"

# ── Filter settings ───────────────────────────────────────────────────────────
SSR_CDS_MAX_FRACTION  = 0.60   # SSR must be in first X% of CDS
UPSTREAM_MAX_DIST_BP  = -200   # SSR must be within 200 bp upstream
APPLY_FRAMESHIFT      = True   # Exclude tri/hex repeat units (len % 3 == 0)

print("Paths set.")
print(f"  PROJECT_ROOT           : {PROJECT_ROOT}")
print(f"  CDS position threshold : {SSR_CDS_MAX_FRACTION*100:.0f}%")
print(f"  Upstream distance cap  : {UPSTREAM_MAX_DIST_BP} bp")
print(f"  Frameshift filter      : {APPLY_FRAMESHIFT}")

## 2. load source tables


In [ ]:
tracts  = pd.read_csv(TRACT_PATH,  low_memory=False)
groups  = pd.read_csv(GROUP_PATH,  low_memory=False)
eggnog  = pd.read_csv(EGGNOG_PATH, sep="\t", low_memory=False)

print(f"Tracts  : {len(tracts):,} rows")
print(f"Groups  : {len(groups):,} rows")
print(f"EggNOG  : {len(eggnog):,} rows")
print()
print("Tract columns  :", tracts.columns.tolist())
print("Group columns  :", groups.columns.tolist())
print("EggNOG columns :", eggnog.columns.tolist())

## 3. build master_strains and add filter flags

no rows are dropped. each filter is recorded as a boolean column.
`passes_all_filters` is the and of all applicable flags.
intragenic rows with missing `tract_aa_pos` are conservatively set to `False`
for `f_cds_position` and `passes_all_filters`, since the position cannot be
verified.


In [ ]:
ms = tracts.copy()

is_intragenic = ms["pv_class"].eq("intragenic")
is_upstream   = ms["pv_class"].eq("upstream")

# ── Flag 1: not downstream ────────────────────────────────────────────────────
ms["f_not_downstream"] = ms["pv_class"].ne("downstream")

# ── Flag 2: upstream distance (only meaningful for upstream rows) ─────────────
# True for non-upstream rows (filter doesn't apply to them)
ms["f_upstream_dist"] = True
ms.loc[is_upstream, "f_upstream_dist"] = (
    ms.loc[is_upstream, "offset_from_gene"].ge(UPSTREAM_MAX_DIST_BP)
)

# ── Flag 3: frameshift-capable unit (only meaningful for intragenic rows) ──────
# True for non-intragenic rows (filter doesn't apply to them)
ms["f_frameshift"] = True
if APPLY_FRAMESHIFT:
    unit_len = ms["tract_unit"].astype(str).str.len()
    ms.loc[is_intragenic, "f_frameshift"] = unit_len[is_intragenic].mod(3).ne(0)

# ── Flag 4: SSR position within first X% of CDS (intragenic only) ─────────────
# Non-intragenic rows: True (filter doesn't apply)
# Intragenic rows with no position data: False (conservative — unverifiable = fail)
# Intragenic rows with position data: check the fraction
ms["f_cds_position"] = True  # default for non-intragenic

has_pos     = ms["tract_aa_pos"].notna() & ms["gene_length_aa"].gt(0)
cds_frac    = ms["tract_aa_pos"] / ms["gene_length_aa"]

# Intragenic with position data: apply threshold
ms.loc[is_intragenic & has_pos,  "f_cds_position"] = cds_frac[is_intragenic & has_pos].le(SSR_CDS_MAX_FRACTION)
# Intragenic without position data: conservative fail
ms.loc[is_intragenic & ~has_pos, "f_cds_position"] = False

# ── Combined flag ─────────────────────────────────────────────────────────────
ms["passes_all_filters"] = (
    ms["f_not_downstream"] &
    ms["f_upstream_dist"]  &
    ms["f_frameshift"]     &
    ms["f_cds_position"]
)

# ── Join COG annotation from eggnog ───────────────────────────────────────────
# Keep only the columns useful at this level; drop duplicates on the join key
eggnog_cols = ["genus", "group_num", "COG_function", "COG_category_primary", "annotated"]
eggnog_cols_present = [c for c in eggnog_cols if c in eggnog.columns]

eggnog_slim = (
    eggnog[eggnog_cols_present]
    .drop_duplicates(subset=["genus", "group_num"])
)
# Ensure group_num types match
ms["group_num"]          = pd.to_numeric(ms["group_num"],          errors="coerce")
eggnog_slim["group_num"] = pd.to_numeric(eggnog_slim["group_num"], errors="coerce")

ms = ms.merge(eggnog_slim, on=["genus", "group_num"], how="left")

print(f"master_strains: {len(ms):,} rows x {ms.shape[1]} columns")
print()
print("Filter pass rates:")
for col in ["f_not_downstream", "f_upstream_dist", "f_frameshift", "f_cds_position", "passes_all_filters"]:
    n      = ms[col].sum()
    total  = len(ms)
    print(f"  {col:22s}: {n:>7,} / {total:,}  ({n/total*100:.1f}%)")

print()
print("Passing by domain:")
print(
    ms.groupby("domain")["passes_all_filters"]
    .agg(total="count", passing="sum")
    .assign(pct=lambda d: (d["passing"]/d["total"]*100).round(1))
    .to_string()
)

## 4. build master_groups and join cog annotations

a group `passes_filter` if at least one of its strain-level rows passes all
filters. filtered pv counts (`pv_in_gene_filtered`, `total_pv_filtered`) are
recorded alongside the original unfiltered counts so both are available in
one place.


In [ ]:
mg = groups.copy()
mg["group_num"] = pd.to_numeric(mg["group_num"], errors="coerce")

# ── Compute filtered PV counts from master_strains ────────────────────────────
# For each (genus, group_num): how many strains pass all filters per pv_class
filtered_counts = (
    ms[ms["passes_all_filters"]]
    .groupby(["genus", "group_num", "pv_class"])
    .size()
    .reset_index(name="n")
)

# Pivot to get one column per pv_class
counts_pivot = filtered_counts.pivot_table(
    index=["genus", "group_num"],
    columns="pv_class",
    values="n",
    fill_value=0
).reset_index()

counts_pivot.columns.name = None

# Rename to filtered versions
rename_map = {}
if "intragenic" in counts_pivot.columns:
    rename_map["intragenic"] = "pv_in_gene_filtered"
if "upstream" in counts_pivot.columns:
    rename_map["upstream"] = "regulatory_pv_filtered"
counts_pivot = counts_pivot.rename(columns=rename_map)

# Add total_pv_filtered
pv_cols_filtered = [c for c in ["pv_in_gene_filtered", "regulatory_pv_filtered"] if c in counts_pivot.columns]
counts_pivot["total_pv_filtered"] = counts_pivot[pv_cols_filtered].sum(axis=1)

# passes_filter: True if total_pv_filtered > 0
counts_pivot["passes_filter"] = counts_pivot["total_pv_filtered"].gt(0)

# ── Join filtered counts onto groups ─────────────────────────────────────────
mg = mg.merge(counts_pivot, on=["genus", "group_num"], how="left")

# Fill NaN counts (groups with no strains in filtered tract data) with 0/False
for col in [c for c in counts_pivot.columns if c not in ["genus", "group_num"]]:
    if col == "passes_filter":
        mg[col] = mg[col].fillna(False)
    else:
        mg[col] = mg[col].fillna(0).astype(int)

# ── Join COG annotation ────────────────────────────────────────────────────────
mg = mg.merge(eggnog_slim, on=["genus", "group_num"], how="left")

print(f"master_groups: {len(mg):,} rows x {mg.shape[1]} columns")
print()
print("Groups passing filter by domain+genus:")
summary = (
    mg.groupby(["domain", "genus"])
    .agg(
        total_groups=("group_num", "count"),
        groups_passing=("passes_filter", "sum")
    )
    .assign(pct=lambda d: (d["groups_passing"]/d["total_groups"]*100).round(1))
)
print(summary.to_string())

## 5. sanity checks


In [ ]:
print("=== master_strains ===")
print(f"Total rows          : {len(ms):,}")
print(f"Domains             : {ms['domain'].unique().tolist()}")
print(f"Genera              : {sorted(ms['genus'].unique().tolist())}")
print(f"passes_all_filters  : {ms['passes_all_filters'].sum():,} ({ms['passes_all_filters'].mean()*100:.1f}%)")
print(f"COG annotated (arch): {ms[ms['domain']=='Archaea']['annotated'].eq(True).sum():,} rows")
print()

print("=== master_groups ===")
print(f"Total rows          : {len(mg):,}")
print(f"passes_filter       : {mg['passes_filter'].sum():,} ({mg['passes_filter'].mean()*100:.1f}%)")
print(f"COG annotated (arch): {mg[mg['domain']=='Archaea']['annotated'].eq(True).sum():,} rows")
print()

# Check: unfiltered total_pv should always be >= filtered
bad = mg[mg["total_pv_filtered"] > mg["total_pv"]]
print(f"Rows where filtered count > unfiltered count: {len(bad)}  (expect 0)")

# Sample
print()
print("Sample — intragenic groups with both filter states:")
sample_cols = ["domain", "genus", "group_num", "likely_function",
               "pv_in_gene", "pv_in_gene_filtered", "passes_filter", "COG_function"]
sample_cols_present = [c for c in sample_cols if c in mg.columns]
display(mg[mg["pv_in_gene"] > 0][sample_cols_present].head(10))

## 6. save


In [ ]:
ms.to_csv(MASTER_STRAINS_PATH, index=False)
mg.to_csv(MASTER_GROUPS_PATH,  index=False)

print(f"Saved master_strains : {MASTER_STRAINS_PATH}")
print(f"  {len(ms):,} rows x {ms.shape[1]} columns")
print()
print(f"Saved master_groups  : {MASTER_GROUPS_PATH}")
print(f"  {len(mg):,} rows x {mg.shape[1]} columns")
print()
print("Column list — master_strains:")
print(ms.columns.tolist())
print()
print("Column list — master_groups:")
print(mg.columns.tolist())